# Stage 3 — Model Improvement: Ensemble + Allocation Search

**PMLDL 2026 project — Minimal Requirement 3.** Extends the Stage 2
[`proposed_hybrid_model`](../proposed_model/proposed_hybrid_model.ipynb) in two
independent directions: (1) an **ensemble of model families** instead of a
single LightGBM, and (2) a systematic **comparison of allocation/sizing
rules** instead of committing to one policy up front.

## Attribution

| Borrowed | Source |
| --- | --- |
| Feature blocks (domain signals, cross terms, lag/rolling context) | unchanged from Stage 2; ultimately from the 4th place write-up and the 61st place notebook (see `proposed_model/proposed_hybrid_model.ipynb` for the full attribution table) |
| Inverse-volatility blending, applied here to *model predictions* rather than raw indicators | 4th place write-up, *"technical model, no learning, short-term mean reversion"* (`baselines/04_4th_writeup.md`) |
| Volatility-targeting overlay (`vol_target_allocation`) | same 4th place write-up |
| Binary discrete-policy allocation | 61st place notebook, <https://www.kaggle.com/code/rafanikitas/hull-eda-training-pipeline> |
| Naive linear allocation | 100th place write-up's metric-anchored sizing philosophy |

No author names, usernames or team identifiers appear anywhere in this
project. Seed is fixed at 42 project-wide; folds are the same `get_folds()`
defaults used by every other notebook, so every number below is directly
comparable to Stage 1/2.

## What "improvement" means here

Stage 2 already showed the domain block earns a small amount of Spearman IC
inside a single LightGBM. This notebook does not re-litigate that — it keeps
the same feature set and instead asks two questions, deliberately kept
separate per the project's tuning conventions (`CLAUDE.md`):

1. **Model stage — does an ensemble beat one model?** Three independent model
   families (LightGBM, CatBoost, Ridge) are each tuned on **mean Spearman IC**
   across the shared purged folds, then combined. The 4th place write-up's
   inverse-volatility blending is repurposed here: instead of blending several
   hand-built indicators, it blends the *predictions of three model
   families*, which is exactly the "smooth out an unstable/sparse signal"
   role that write-up describes for its auxiliary indicators.
2. **Sizing stage — which allocation rule extracts the most modified Sharpe
   from the same ensemble prediction?** Four rules are compared:
   `naive_allocation` (continuous tilt), `binary_allocation` (61st place,
   discrete two-state), `ternary_allocation` (a three-state extension added
   in `src/allocation.py` for this notebook), and `vol_target_allocation`
   (4th place volatility overlay). Each rule's free parameter(s) are searched
   *against modified Sharpe*, never against Spearman IC — sizing controls
   exposure and volatility, which IC cannot see. Search budgets are
   deliberately modest (a few dozen trials per rule, ~120 total) and the
   objective penalises cross-fold std (`mean - 0.25 * std`) so the search
   cannot lock onto one favourable fold.

### Leakage

Same guarantees as Stage 2: every feature is built from `past_returns()`,
folds are purged (`purge=1`) and embargoed (`embargo=20`), and the last 180
`date_id`s are held out and touched exactly once, at the very end.

In [1]:
# ============================ SETUP ============================
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import optuna

from src import *          # shared interface, see INTERFACE.md

optuna.logging.set_verbosity(optuna.logging.WARNING)

set_seed()                 # SEED = 42, fixed project-wide
print("seed:", SEED, "| target:", TARGET)

# Trial budgets. QUICK=True is for a fast smoke run; the reported results use
# QUICK=False. Model-family budgets are intentionally larger than the
# allocation/policy budget below (see markdown discussion above).
QUICK = True
N_TRIALS_LGBM = 10 if QUICK else 40
N_TRIALS_CAT = 8 if QUICK else 25
N_TRIALS_RIDGE = 8 if QUICK else 30
N_TRIALS_ALLOC = 10 if QUICK else 30   # per allocation rule -> ~120 total


C:\Users\bobs\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


seed: 42 | target: market_forward_excess_returns


## 1. Data

Identical call to every other notebook — see `INTERFACE.md`.

In [2]:
hull = load_dataset()
print(hull)
print("train date_id:", hull.train[DATE_COL].min(), "->", hull.train[DATE_COL].max())
print("public date_id:", hull.public[DATE_COL].min(), "->", hull.public[DATE_COL].max())


HullData(train=(7862, 98), public=(180, 98), n_raw_features=94)
train date_id: 1006 -> 8867
public date_id: 8868 -> 9047


## 2. Feature engineering

Same three blocks as Stage 2, unchanged, so any difference in the numbers
below is attributable to the model/allocation stage, not to a different
feature set. See `proposed_model/proposed_hybrid_model.ipynb` for the full
description of each block.

In [3]:
LAG_ROLL_COLUMNS = ["M4", "V13", "S5", "S2", "D2", "E19", "P7", "P6",
                    "P3", "P13", "P4", "P5", "M2", "V5"]
LAG_ROLL_COLUMNS = [c for c in LAG_ROLL_COLUMNS if c in hull.full.columns]

feat_df, FEATURES = build_features(
    hull.full,
    lag_roll_columns=LAG_ROLL_COLUMNS,   # Block C: raw anonymised columns only
    price_features=True,                 # Block A: 4th place domain signals
    cross_terms=True,                    # Block B
)

assert not set(FEATURES) & set(LOOKAHEAD_COLS), "look-ahead column in feature list"
print(f"total features: {len(FEATURES)}")


total features: 335


In [4]:
cut = feat_df[DATE_COL].max() - PUBLIC_TEST_SIZE
train_df = feat_df[feat_df[DATE_COL] <= cut].reset_index(drop=True)
public_df = feat_df[feat_df[DATE_COL] > cut].reset_index(drop=True)

train_df, public_df = impute(train_df, public_df, columns=FEATURES)
assert train_df[FEATURES].isna().sum().sum() == 0
assert public_df[FEATURES].isna().sum().sum() == 0
print("train:", train_df.shape, "| public:", public_df.shape)


train: (7862, 339) | public: (180, 339)


## 3. Validation

Same purged, embargoed folds as every other notebook: 4 expanding-window
splits, `purge=1` and `embargo=20` trading days.

In [5]:
folds = get_folds(train_df)          # defaults — do not override
assert_no_leakage(folds)
display(describe_folds(train_df, folds))

X = train_df[FEATURES]
y = train_df[TARGET]


,fold,n_train,n_val,train_end_date_id,val_start_date_id,val_end_date_id,gap_days
0,0,1571,1552,2576,2598,4149,22
1,1,3143,1552,4148,4170,5721,22
2,2,4715,1552,5720,5742,7293,22
3,3,6287,1554,7292,7314,8867,22


## 4. Three model families, each tuned on mean Spearman IC

Per the project's tuning convention, hyperparameters are searched against
**mean Spearman rank correlation across the shared folds**, never against
`modified_sharpe` — that objective is reserved for the sizing stage below.

* **LightGBM** — same search space as Stage 2.
* **CatBoost** — a second, structurally different GBDT (ordered boosting,
  symmetric trees), tuned independently so its errors are not simply a
  noisier copy of LightGBM's.
* **Ridge** — a linear model on standardised features. Deliberately the
  simplest possible family: if the tree-based models cannot beat a
  regularised linear fit on the same features, that is itself informative,
  and a linear model's errors are structurally the least correlated with
  either GBDT's.

In [6]:
def cv_spearman_lgbm(params):
    scores = []
    for fold in folds:
        X_tr, y_tr = X.iloc[fold.train_idx], y.iloc[fold.train_idx]
        X_va, y_va = X.iloc[fold.val_idx], y.iloc[fold.val_idx]
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_X=X_va, eval_y=y_va, eval_metric="rmse",
                  callbacks=[lgb.early_stopping(100, verbose=False),
                             lgb.log_evaluation(-1)])
        scores.append(spearman_ic(y_va.to_numpy(), model.predict(X_va)))
    return float(np.mean(scores))


def lgbm_objective(trial):
    params = {
        "objective": "regression", "metric": "rmse",
        "n_estimators": trial.suggest_int("n_estimators", 300, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.07, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1, "random_state": SEED, "verbosity": -1,
    }
    return cv_spearman_lgbm(params)


lgbm_study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED), study_name="ensemble_lgbm")
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS_LGBM, show_progress_bar=True)

LGBM_PARAMS = {"objective": "regression", "metric": "rmse", "subsample_freq": 1,
               "random_state": SEED, "verbosity": -1, **lgbm_study.best_params}
print(f"LightGBM: {len(lgbm_study.trials)} trials | best mean Spearman IC: {lgbm_study.best_value:.4f}")
print(LGBM_PARAMS)



  0%|          | 0/10 [00:00<?, ?it/s]


Best trial: 0. Best value: 0.0312268:   0%|          | 0/10 [00:39<?, ?it/s]


Best trial: 0. Best value: 0.0312268:  10%|█         | 1/10 [00:39<05:53, 39.24s/it]


Best trial: 1. Best value: 0.0382323:  10%|█         | 1/10 [00:48<05:53, 39.24s/it]


Best trial: 1. Best value: 0.0382323:  20%|██        | 2/10 [00:48<02:51, 21.47s/it]

...


Best trial: 5. Best value: 0.048578:  90%|█████████ | 9/10 [02:02<00:07,  7.70s/it]


Best trial: 5. Best value: 0.048578:  90%|█████████ | 9/10 [02:18<00:07,  7.70s/it]


Best trial: 5. Best value: 0.048578: 100%|██████████| 10/10 [02:18<00:00, 10.28s/it]


Best trial: 5. Best value: 0.048578: 100%|██████████| 10/10 [02:18<00:00, 13.83s/it]

LightGBM: 10 trials | best mean Spearman IC: 0.0486
{'objective': 'regression', 'metric': 'rmse', 'subsample_freq': 1, 'random_state': 42, 'verbosity': -1, 'n_estimators': 629, 'learning_rate': 0.02621036302731938, 'max_depth': 4, 'num_leaves': 235, 'reg_lambda': 0.010842262717330166, 'reg_alpha': 0.4467752817973907, 'colsample_bytree': 0.7246844304357644, 'subsample': 0.8080272084711243}


In [7]:
def cv_spearman_cat(params):
    scores = []
    for fold in folds:
        X_tr, y_tr = X.iloc[fold.train_idx], y.iloc[fold.train_idx]
        X_va, y_va = X.iloc[fold.val_idx], y.iloc[fold.val_idx]
        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va),
                  early_stopping_rounds=100, verbose=False)
        scores.append(spearman_ic(y_va.to_numpy(), model.predict(X_va)))
    return float(np.mean(scores))


def cat_objective(trial):
    params = {
        "loss_function": "RMSE",
        "iterations": trial.suggest_int("iterations", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "random_seed": SEED, "verbose": False, "allow_writing_files": False,
        "bootstrap_type": "Bernoulli",
    }
    return cv_spearman_cat(params)


cat_study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED), study_name="ensemble_catboost")
cat_study.optimize(cat_objective, n_trials=N_TRIALS_CAT, show_progress_bar=True)

CAT_PARAMS = {"loss_function": "RMSE", "random_seed": SEED, "verbose": False,
              "allow_writing_files": False, "bootstrap_type": "Bernoulli",
              **cat_study.best_params}
print(f"CatBoost: {len(cat_study.trials)} trials | best mean Spearman IC: {cat_study.best_value:.4f}")
print(CAT_PARAMS)



  0%|          | 0/8 [00:00<?, ?it/s]


Best trial: 0. Best value: 0.0436103:   0%|          | 0/8 [00:30<?, ?it/s]


Best trial: 0. Best value: 0.0436103:  12%|█▎        | 1/8 [00:30<03:30, 30.10s/it]


Best trial: 0. Best value: 0.0436103:  12%|█▎        | 1/8 [01:57<03:30, 30.10s/it]


Best trial: 0. Best value: 0.0436103:  25%|██▌       | 2/8 [01:57<06:23, 63.92s/it]

...


Best trial: 3. Best value: 0.055534:  88%|████████▊ | 7/8 [03:50<00:22, 22.60s/it]


Best trial: 3. Best value: 0.055534:  88%|████████▊ | 7/8 [03:58<00:22, 22.60s/it]


Best trial: 3. Best value: 0.055534: 100%|██████████| 8/8 [03:58<00:00, 17.90s/it]


Best trial: 3. Best value: 0.055534: 100%|██████████| 8/8 [03:58<00:00, 29.77s/it]

CatBoost: 8 trials | best mean Spearman IC: 0.0555
{'loss_function': 'RMSE', 'random_seed': 42, 'verbose': False, 'allow_writing_files': False, 'bootstrap_type': 'Bernoulli', 'iterations': 520, 'learning_rate': 0.02014847788415866, 'depth': 6, 'l2_leaf_reg': 0.19762189340280073, 'subsample': 0.7164916560792167}


In [8]:
def cv_spearman_ridge(alpha):
    scores = []
    for fold in folds:
        X_tr, y_tr = X.iloc[fold.train_idx], y.iloc[fold.train_idx]
        X_va, y_va = X.iloc[fold.val_idx], y.iloc[fold.val_idx]
        scaler = StandardScaler().fit(X_tr)
        model = Ridge(alpha=alpha, random_state=SEED)
        model.fit(scaler.transform(X_tr), y_tr)
        scores.append(spearman_ic(y_va.to_numpy(), model.predict(scaler.transform(X_va))))
    return float(np.mean(scores))


def ridge_objective(trial):
    alpha = trial.suggest_float("alpha", 1e-2, 1e4, log=True)
    return cv_spearman_ridge(alpha)


ridge_study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED), study_name="ensemble_ridge")
ridge_study.optimize(ridge_objective, n_trials=N_TRIALS_RIDGE, show_progress_bar=True)

RIDGE_ALPHA = ridge_study.best_params["alpha"]
print(f"Ridge: {len(ridge_study.trials)} trials | best mean Spearman IC: "
      f"{ridge_study.best_value:.4f} | alpha={RIDGE_ALPHA:.4g}")



  0%|          | 0/8 [00:00<?, ?it/s]


Best trial: 0. Best value: 0.0114207:   0%|          | 0/8 [00:00<?, ?it/s]


Best trial: 0. Best value: 0.0114207:  12%|█▎        | 1/8 [00:00<00:02,  2.50it/s]


Best trial: 1. Best value: 0.0480071:  12%|█▎        | 1/8 [00:00<00:02,  2.50it/s]


Best trial: 1. Best value: 0.0480071:  25%|██▌       | 2/8 [00:00<00:02,  2.39it/s]

...


Best trial: 1. Best value: 0.0480071:  88%|████████▊ | 7/8 [00:02<00:00,  2.39it/s]


Best trial: 1. Best value: 0.0480071:  88%|████████▊ | 7/8 [00:03<00:00,  2.39it/s]


Best trial: 1. Best value: 0.0480071: 100%|██████████| 8/8 [00:03<00:00,  2.42it/s]


Best trial: 1. Best value: 0.0480071: 100%|██████████| 8/8 [00:03<00:00,  2.41it/s]

Ridge: 8 trials | best mean Spearman IC: 0.0480 | alpha=5062


## 5. Out-of-fold predictions and blend

Each model family is refit per fold with its tuned hyperparameters (never
inside the Optuna search itself, to keep the reported OOF honest). The three
OOF prediction columns are then combined two ways:

* **simple average** — the naive ensemble baseline;
* **inverse-volatility blend** (`blend_signals`, the same helper the
  no-learning benchmark in Stage 2 used) — each model's column is weighted by
  the reciprocal of its own rolling volatility, so an unstable/sparse model is
  automatically down-weighted rather than hurting the blend at a fixed 1/3
  weight.

The blend is chosen by mean Spearman IC — still the model-stage objective,
not `modified_sharpe`.

In [9]:
oof_rows = []
for fold in folds:
    X_tr, y_tr = X.iloc[fold.train_idx], y.iloc[fold.train_idx]
    X_va, y_va = X.iloc[fold.val_idx], y.iloc[fold.val_idx]
    val = train_df.iloc[fold.val_idx]

    lgbm = lgb.LGBMRegressor(**LGBM_PARAMS)
    lgbm.fit(X_tr, y_tr, eval_X=X_va, eval_y=y_va, eval_metric="rmse",
             callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])

    cat = CatBoostRegressor(**CAT_PARAMS)
    cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=100, verbose=False)

    scaler = StandardScaler().fit(X_tr)
    ridge = Ridge(alpha=RIDGE_ALPHA, random_state=SEED)
    ridge.fit(scaler.transform(X_tr), y_tr)

    oof_rows.append(pd.DataFrame({
        DATE_COL: val[DATE_COL].to_numpy(),
        "fold": fold.index,
        "y_true": y_va.to_numpy(),
        "forward_returns": val["forward_returns"].to_numpy(),
        "risk_free_rate": val["risk_free_rate"].to_numpy(),
        "vol_20": val["vol_20"].to_numpy(),
        "lgbm": lgbm.predict(X_va),
        "catboost": cat.predict(X_va),
        "ridge": ridge.predict(scaler.transform(X_va)),
    }))

oof = pd.concat(oof_rows, ignore_index=True)
MODEL_COLS = ["lgbm", "catboost", "ridge"]

oof["blend_avg"] = oof[MODEL_COLS].mean(axis=1)
oof["blend_ivol"] = blend_signals(oof[MODEL_COLS]).fillna(0.0)

for col in MODEL_COLS + ["blend_avg", "blend_ivol"]:
    per_fold = [spearman_ic(oof.loc[oof.fold == f.index, "y_true"],
                             oof.loc[oof.fold == f.index, col]) for f in folds]
    print(f"{col:>10s}: mean Spearman IC = {np.mean(per_fold):+.4f}  (per fold: "
          + ", ".join(f"{s:+.4f}" for s in per_fold) + ")")


      lgbm: mean Spearman IC = +0.0486  (per fold: +0.0611, +0.0313, +0.0404, +0.0615)
  catboost: mean Spearman IC = +0.0555  (per fold: +0.0577, +0.0495, +0.0560, +0.0589)
     ridge: mean Spearman IC = +0.0480  (per fold: +0.0610, +0.0468, +0.0292, +0.0550)
 blend_avg: mean Spearman IC = +0.0609  (per fold: +0.0737, +0.0469, +0.0513, +0.0718)
blend_ivol: mean Spearman IC = +0.0595  (per fold: +0.0745, +0.0408, +0.0490, +0.0737)


In [10]:
ic_avg = np.mean([spearman_ic(oof.loc[oof.fold == f.index, "y_true"],
                              oof.loc[oof.fold == f.index, "blend_avg"]) for f in folds])
ic_ivol = np.mean([spearman_ic(oof.loc[oof.fold == f.index, "y_true"],
                               oof.loc[oof.fold == f.index, "blend_ivol"]) for f in folds])
ENSEMBLE_COL = "blend_ivol" if ic_ivol >= ic_avg else "blend_avg"
print(f"chosen ensemble: {ENSEMBLE_COL}  (inverse-vol IC={ic_ivol:+.4f} vs average IC={ic_avg:+.4f})")


chosen ensemble: blend_avg  (inverse-vol IC=+0.0595 vs average IC=+0.0609)


## 6. Allocation-rule search

The ensemble's OOF prediction (`oof[ENSEMBLE_COL]`) is now fixed. Four
allocation rules are each given a small, independent Optuna search
(`N_TRIALS_ALLOC` trials, ~30 by default, ~120 total) against
**`modified_sharpe`**, scored **per fold then aggregated** as
`mean - 0.25 * std` — never pooling rows across folds, and penalising a rule
that only works in one favourable fold. This is the sizing-stage objective;
nothing here touches Spearman IC.

* `naive_allocation(pred, k)` — continuous tilt around the passive `w = 1`.
* `binary_allocation(pred, threshold)` — 61st place discrete two-state policy.
* `ternary_allocation(pred, threshold)` — three-state extension added for this
  notebook (`src/allocation.py`): risk-free / passive / 2x-leveraged.
* `vol_target_allocation(pred, vol_20, target_vol=0.12, k)` — 4th place
  volatility-targeting overlay, using the causal `vol_20` column already in
  the feature frame.

Threshold searches are scaled to the ensemble's own OOF prediction std so the
search range is sensible regardless of which model/blend ends up dominating.

In [11]:
PRED_STD = float(oof[ENSEMBLE_COL].std())
print(f"ensemble OOF prediction std: {PRED_STD:.5f}  (threshold searches scaled to this)")


def fold_scores(weight_fn):
    """modified_sharpe per fold for a weight function of the ensemble OOF signal."""
    scores = []
    for f in folds:
        sub = oof.loc[oof.fold == f.index]
        w = weight_fn(sub[ENSEMBLE_COL].to_numpy(), sub)
        scores.append(modified_sharpe(w, sub["forward_returns"].to_numpy(),
                                       sub["risk_free_rate"].to_numpy()))
    return np.array(scores)


def alloc_score(scores):
    """mean - 0.25*std across folds, penalising a rule that only wins one fold."""
    return float(np.mean(scores) - 0.25 * np.std(scores))


def _run_alloc_study(name, objective):
    study = optuna.create_study(direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED), study_name=f"alloc_{name}")
    study.optimize(objective, n_trials=N_TRIALS_ALLOC, show_progress_bar=False)
    return study


def naive_objective(trial):
    k = trial.suggest_float("k", 0.1, 2000.0, log=True)
    return alloc_score(fold_scores(lambda pred, sub, k=k: naive_allocation(pred, k=k)))

naive_study = _run_alloc_study("naive", naive_objective)
NAIVE_K = naive_study.best_params["k"]


def binary_objective(trial):
    t = trial.suggest_float("threshold", -3 * PRED_STD, 3 * PRED_STD)
    return alloc_score(fold_scores(lambda pred, sub, t=t: binary_allocation(pred, threshold=t)))

binary_study = _run_alloc_study("binary", binary_objective)
BINARY_T = binary_study.best_params["threshold"]


def ternary_objective(trial):
    t = trial.suggest_float("threshold", 0.0, 3 * PRED_STD)
    return alloc_score(fold_scores(lambda pred, sub, t=t: ternary_allocation(pred, threshold=t)))

ternary_study = _run_alloc_study("ternary", ternary_objective)
TERNARY_T = ternary_study.best_params["threshold"]


def vol_target_objective(trial):
    k = trial.suggest_float("k", 0.1, 2000.0, log=True)
    return alloc_score(fold_scores(
        lambda pred, sub, k=k: vol_target_allocation(pred, sub["vol_20"].to_numpy(), target_vol=0.12, k=k)))

vol_study = _run_alloc_study("vol_target", vol_target_objective)
VOL_K = vol_study.best_params["k"]

print(f"naive:       k={NAIVE_K:.4g}              score={naive_study.best_value:+.4f}")
print(f"binary:      threshold={BINARY_T:+.5f}      score={binary_study.best_value:+.4f}")
print(f"ternary:     threshold={TERNARY_T:.5f}       score={ternary_study.best_value:+.4f}")
print(f"vol-target:  k={VOL_K:.4g}              score={vol_study.best_value:+.4f}")


ensemble OOF prediction std: 0.00090  (threshold searches scaled to this)


naive:       k=531.4              score=+0.3905
binary:      threshold=+0.00055      score=+0.4583
ternary:     threshold=0.00162       score=+0.3899
vol-target:  k=531.4              score=+0.5200


## 7. Cross-validated comparison of the four rules

In [12]:
RULES = {
    "naive": lambda pred, sub: naive_allocation(pred, k=NAIVE_K),
    "binary": lambda pred, sub: binary_allocation(pred, threshold=BINARY_T),
    "ternary": lambda pred, sub: ternary_allocation(pred, threshold=TERNARY_T),
    "vol_target": lambda pred, sub: vol_target_allocation(pred, sub["vol_20"].to_numpy(),
                                                          target_vol=0.12, k=VOL_K),
}

rows = []
for name, weight_fn in RULES.items():
    for f in folds:
        sub = oof.loc[oof.fold == f.index]
        w = weight_fn(sub[ENSEMBLE_COL].to_numpy(), sub)
        m = evaluate(sub["y_true"].to_numpy(), sub[ENSEMBLE_COL].to_numpy(), weights=w,
                     forward_returns=sub["forward_returns"].to_numpy(),
                     risk_free_rate=sub["risk_free_rate"].to_numpy())
        m["rule"] = name
        m["fold"] = f.index
        rows.append(m)

rule_fold_table = pd.DataFrame(rows)
display(rule_fold_table[["rule", "fold", "spearman_ic", "modified_sharpe", "sharpe",
                          "vol_ratio", "mean_weight"]].round(4))

rule_summary = (rule_fold_table.groupby("rule")
                .agg(modified_sharpe_mean=("modified_sharpe", "mean"),
                     modified_sharpe_std=("modified_sharpe", lambda s: s.std(ddof=0)),
                     sharpe_mean=("sharpe", "mean"),
                     vol_ratio_mean=("vol_ratio", "mean"),
                     mean_weight=("mean_weight", "mean"))
                .assign(score=lambda d: d.modified_sharpe_mean - 0.25 * d.modified_sharpe_std)
                .sort_values("score", ascending=False))
display(rule_summary.round(4))

BEST_RULE = rule_summary.index[0]
print(f"best rule by mean - 0.25*std: {BEST_RULE}")


,rule,fold,spearman_ic,modified_sharpe,sharpe,vol_ratio,mean_weight
0,naive,0,0.0737,-0.0353,-0.0508,1.6366,1.3601
1,naive,1,0.0469,0.2455,0.3015,1.4281,1.0891
2,naive,2,0.0513,1.0514,1.0514,1.0943,0.8044
3,naive,3,0.0718,0.7197,0.7814,1.2858,0.9708
4,binary,0,0.0737,0.3351,0.3351,0.8651,0.5142
5,binary,1,0.0469,0.5261,0.5261,0.7347,0.2545
6,binary,2,0.0513,0.4551,0.6981,0.4865,0.0883
7,binary,3,0.0718,0.6211,0.7594,0.6091,0.1692
8,ternary,0,0.0737,-0.0024,-0.0030,1.4484,1.1398
9,ternary,1,0.0469,0.1916,0.2043,1.2660,1.0483


,modified_sharpe_mean,modified_sharpe_std,sharpe_mean,vol_ratio_mean,mean_weight,score
rule,,,,,,
vol_target,0.6210,0.4041,0.6210,0.9416,0.9430,0.5200
binary,0.4844,0.1044,0.5797,0.6739,0.2565,0.4583
naive,0.4953,0.4194,0.5209,1.3612,1.0561,0.3905
ternary,0.5007,0.4432,0.5037,1.2445,1.0535,0.3899


best rule by mean - 0.25*std: vol_target


In [13]:
best_fold_metrics = [
    {k: v for k, v in row.items() if k not in ("rule", "fold")}
    for row in rows if row["rule"] == BEST_RULE
]
cv_metrics = aggregate_folds(best_fold_metrics)
print(f"CV metrics for '{BEST_RULE}':")
print("mean Spearman IC: {spearman_ic_mean:.4f} (sd {spearman_ic_std:.4f})".format(**cv_metrics))
print("mean modified Sharpe: {modified_sharpe_mean:.4f} (sd {modified_sharpe_std:.4f})".format(**cv_metrics))


CV metrics for 'vol_target':
mean Spearman IC: 0.0609 (sd 0.0119)
mean modified Sharpe: 0.6210 (sd 0.4041)


## 8. Held-out public block

Each model is refit once on the full training period with its tuned
hyperparameters, ensembled the same way as the CV stage, then every
allocation rule (with the parameters already fixed above — nothing further is
tuned on this block) is scored once on the 180 held-out `date_id`s.

In [14]:
final_lgbm = lgb.LGBMRegressor(**LGBM_PARAMS)
final_lgbm.fit(X, y)

final_cat = CatBoostRegressor(**CAT_PARAMS)
final_cat.fit(X, y, verbose=False)

final_scaler = StandardScaler().fit(X)
final_ridge = Ridge(alpha=RIDGE_ALPHA, random_state=SEED)
final_ridge.fit(final_scaler.transform(X), y)

pub_preds = pd.DataFrame({
    "lgbm": final_lgbm.predict(public_df[FEATURES]),
    "catboost": final_cat.predict(public_df[FEATURES]),
    "ridge": final_ridge.predict(final_scaler.transform(public_df[FEATURES])),
})
pub_preds["blend_avg"] = pub_preds[MODEL_COLS].mean(axis=1)
pub_preds["blend_ivol"] = blend_signals(pub_preds[MODEL_COLS]).fillna(0.0)
pub_pred = pub_preds[ENSEMBLE_COL].to_numpy()

pub_rows = []
for name, weight_fn in RULES.items():
    w = weight_fn(pub_pred, public_df)
    m = modified_sharpe(w, public_df["forward_returns"].to_numpy(),
                        public_df["risk_free_rate"].to_numpy(), return_components=True)
    m["rule"] = name
    m["mean_weight"] = float(np.mean(w))
    m["spearman_ic"] = spearman_ic(public_df[TARGET].to_numpy(), pub_pred)
    pub_rows.append(m)

pub_table = pd.DataFrame(pub_rows)[["rule", "spearman_ic", "modified_sharpe", "sharpe",
                                     "vol_ratio", "vol_penalty", "return_penalty", "mean_weight"]]
display(pub_table.round(4))

public_metrics = evaluate(
    public_df[TARGET].to_numpy(), pub_pred, weights=RULES[BEST_RULE](pub_pred, public_df),
    forward_returns=public_df["forward_returns"].to_numpy(),
    risk_free_rate=public_df["risk_free_rate"].to_numpy(),
)
print(f"headline rule (\'{BEST_RULE}\') on the held-out block:")
print({k: round(v, 4) for k, v in public_metrics.items() if isinstance(v, float)})


,rule,spearman_ic,modified_sharpe,sharpe,vol_ratio,vol_penalty,return_penalty,mean_weight
0,naive,0.1721,1.1120,1.4549,1.5083,1.3083,1.0000,1.0600
1,binary,0.1721,1.2417,1.2931,0.7748,1.0000,1.0414,0.2778
2,ternary,0.1721,0.8892,1.1352,1.4767,1.2767,1.0000,1.0833
3,vol_target,0.1721,1.4610,1.4610,1.1533,1.0000,1.0000,0.9884


headline rule ('vol_target') on the held-out block:
{'rmse': 0.0102, 'r2': 0.0207, 'spearman_ic': 0.1721, 'hit_rate': 0.5278, 'modified_sharpe': 1.461, 'sharpe': 1.461, 'vol_ratio': 1.1533, 'vol_penalty': 1.0, 'return_penalty': 1.0, 'bounds_violation': 0.0, 'ann_return': 0.3339, 'ann_volatility': 0.1887, 'max_drawdown': -0.1396, 'benchmark_sharpe': 1.208, 'mean_weight': 0.9884, 'weight_turnover': 0.2701}


## 9. Save results

In [15]:
MODEL_NAME = "ensemble_lgbm_catboost_ridge"

save_result(model=MODEL_NAME, stage="improved", metrics=cv_metrics, split="cv",
            params={"lgbm": LGBM_PARAMS, "catboost": CAT_PARAMS, "ridge_alpha": RIDGE_ALPHA,
                    "ensemble": ENSEMBLE_COL, "allocation_rule": BEST_RULE,
                    "naive_k": NAIVE_K, "binary_threshold": BINARY_T,
                    "ternary_threshold": TERNARY_T, "vol_target_k": VOL_K},
            notes=f"3-model ensemble ({ENSEMBLE_COL}); Optuna per family on mean Spearman; "
                  f"allocation rules tuned on modified_sharpe (mean-0.25*std); best rule: {BEST_RULE}")
save_result(model=MODEL_NAME, stage="improved", metrics=public_metrics, split="public",
            params={"allocation_rule": BEST_RULE},
            notes="refit on full train period, scored once on held-out 180 rows")

save_predictions(MODEL_NAME, public_df[DATE_COL], public_df[TARGET], pub_pred,
                 RULES[BEST_RULE](pub_pred, public_df))


WindowsPath('C:/Users/bobs/Desktop/Новая папка (3)/PMDL-Project-Market-Prediction/results/artifacts/ensemble_lgbm_catboost_ridge_predictions.csv')

In [ ]:
print("--- cross-validated ---")
display(compare(split="cv"))
print("--- held-out public block ---")
display(compare(split="public"))

--- cross-validated ---


,model,stage,split,spearman_ic_mean,rmse_mean,modified_sharpe_mean,sharpe_mean,vol_ratio_mean
0,ensemble_lgbm_catboost_ridge,improved,cv,0.060923,0.010872,0.621034,0.621034,0.941558
1,improved_ensemble,improved,cv,0.050747,0.937689,0.523536,0.555378,1.263074
2,hybrid_domain_lgbm,proposed,cv,0.075462,0.010853,0.452659,0.452659,1.022154


--- held-out public block ---


,model,stage,split,spearman_ic,rmse,modified_sharpe,sharpe,vol_ratio
0,ensemble_lgbm_catboost_ridge,improved,public,0.172121,0.010206,1.460990,1.460990,1.153299
1,hybrid_domain_lgbm,proposed,public,0.180720,0.010549,1.122170,1.163230,1.236590
2,lgbm_61st,baseline,public,0.047796,0.631812,0.997153,1.185332,0.726148
3,elasticnet_leak_safe,baseline,public,-0.135918,0.121817,0.387571,1.385695,0.104900
4,online_ensemble_100th,baseline,public,0.000086,0.545907,0.274439,0.725019,0.473175
5,improved_ensemble,improved,public,0.043331,1.333474,0.010594,0.047391,1.309702
